# Five-minute processing-time forecasting

Forecast the single reading of `avg_transaction_processing_time_ms` five minutes
from the forecast origin. This is aggregate processing time, not channel-specific
or endpoint latency.

The notebook reviews a saved experiment and recomputes its scores from source
readings. Train from the repository root first:

```bash
.venv/bin/python scripts/train_forecast.py --data-dir data/raw
```

An existing checkpoint is not overwritten unless `--overwrite` is supplied.
The comparison includes baselines, ridge regression, and histogram gradient
boosting, selected using three chronological validation windows. Discovery already
covered the full monitoring range; the test period is reserved for model evaluation
but was not unseen during data exploration.

Notebook outputs are left empty in Git. Select the project's `.venv` kernel and
run all cells after training. Plots are written under `reports/forecast/`.


In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "banking_observability").is_dir():
    raise RuntimeError("Run from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT))
DATA_DIR = Path(os.environ.get("BANKING_DATA_DIR", str(ROOT / "data/raw")))
REPORTS = Path(os.environ.get("BANKING_REPORT_DIR", str(ROOT / "reports/forecast")))
MODELS = Path(os.environ.get("BANKING_MODEL_DIR", str(ROOT / "models/latency_5min")))

from banking_observability.forecast_features import TARGET, FeatureSpec, build_features, make_supervised, read_history
from banking_observability.forecast_experiment import score
from banking_observability.forecasting import LatencyForecaster

train = read_history(DATA_DIR / "infrastructure_metrics_train.csv")
test = read_history(DATA_DIR / "infrastructure_metrics_test.csv")
history = pd.concat([train, test], ignore_index=True)
selection = json.loads((REPORTS / "selection.json").read_text())
summary = json.loads((REPORTS / "summary.json").read_text())
predictions = pd.read_csv(REPORTS / "test_predictions.csv", parse_dates=["forecast_origin", "target_timestamp"])
cv = pd.read_csv(REPORTS / "cv_leaderboard.csv")
fold_scores = pd.read_csv(REPORTS / "cv_fold_metrics.csv")
selected = selection["selected_candidate"]["name"]
best_ridge = cv.loc[cv.kind.eq("ridge")].sort_values(["mean_mae_ms", "mean_rmse_ms", "candidate"]).iloc[0].candidate
print(f"Training readings: {len(train):,}; test readings: {len(test):,}")
print(f"Source target: {TARGET}")
print(f"Validation-selected candidate: {selected}")

## 1. Forecast target

At origin `t`, the latest available reading is `y[t]`; the target is `y[t+5]`.
Trailing windows include `y[t]` but stop before `y[t+1]`.

The example below shows the readings and target for one training row. The daily
baseline uses the target minute on the previous day: `t + 5 - 1440`. Times use the
source clock because the CSVs do not specify a timezone or reporting delay.


In [ ]:
spec = FeatureSpec()
X, y = make_supervised(train, spec)
origin = X.index[100]
indexed_train = train.set_index("timestamp")
example = pd.Series({
    "origin": origin,
    "target_timestamp": origin + pd.Timedelta(minutes=5),
    "latest_observed_ms": X.loc[origin, "latency__lag_0"],
    "trailing_15_minute_mean_ms": X.loc[origin, "latency__mean_15"],
    "daily_baseline_source_timestamp": origin + pd.Timedelta(minutes=5 - 1440),
    "daily_baseline_ms": X.loc[origin, "latency__previous_day_at_target"],
    "actual_five_minutes_later_ms": y.loc[origin],
}, name="one_real_training_example")
assert y.loc[origin] == indexed_train.loc[origin + pd.Timedelta(minutes=5), TARGET]
assert np.isclose(X.loc[origin, "latency__mean_15"], indexed_train.loc[origin - pd.Timedelta(minutes=14):origin, TARGET].mean())
display(example)
print(f"Common feature columns: {X.shape[1]}; eligible training examples: {len(X):,}")

## 2. Validation boundaries

A training row at `t` needs its label from `t+5`. Splitting on forecast origins
alone can leave training labels inside the validation period.

The five-row gap keeps the last training label before the first validation
origin. Scaling and model fitting use only the training rows in each fold.


In [ ]:
folds = pd.read_csv(REPORTS / "folds.csv")
assert (pd.to_datetime(folds.train_target_end) < pd.to_datetime(folds.validation_origin_start)).all()
assert folds.purged_origin_rows.eq(5).all()
display(folds[["fold", "train_rows", "train_target_end", "validation_origin_start", "validation_target_end"]])

## 3. What changed on Saturday?

Compare 09:00–13:59 in the training file. These hours are complete on each of the
five weekdays and on Saturday, unlike the full-day coverage.

The table shows whether processing time and CPU have similar levels during the
same hours. With only part of one weekend in training, this cannot establish a
stable weekly pattern.


In [ ]:
same_hours = train.loc[train.timestamp.dt.hour.between(9, 13)].copy()
same_hours["calendar_group"] = same_hours.timestamp.dt.dayofweek.ge(5).map({False: "Monday–Friday", True: "Saturday"})
display(same_hours.groupby("calendar_group").agg(
    readings=(TARGET, "size"),
    mean_processing_ms=(TARGET, "mean"),
    mean_cpu=("cpu_utilization", "mean"),
))
print("Training latency autocorrelation:")
display(pd.Series({lag: train[TARGET].autocorr(lag) for lag in (1, 5, 15, 60, 1440)}, name="autocorrelation"))

## 4. Validation results

`experiment_plan.json` records five baselines, three ridge penalties, and twelve
boosted-tree configurations. The tree models use clock-only features, latency
history plus clock features, or the full set including resource histories.

Fold 3 includes Saturday. Compare errors across the individual folds before
reading the overall ranking. Selection uses mean validation MAE, with RMSE and
candidate name as tie breakers.


In [ ]:
display(cv[["candidate", "mean_mae_ms", "mean_rmse_ms", "max_fold_mae_ms"]])
comparison_names = [selected, "persistence", "daily_seasonal_naive", best_ridge, *selection["group_best_hgb_candidates"].values()]
comparison_names = list(dict.fromkeys(comparison_names))
by_fold = fold_scores.loc[fold_scores.candidate.isin(comparison_names)].pivot(index="candidate", columns="fold", values="mae_ms")
display(by_fold)
assert selected == cv.sort_values(["mean_mae_ms", "mean_rmse_ms", "candidate"]).iloc[0].candidate

fig, ax = plt.subplots(figsize=(10.5, 4.8))
for name in [selected, best_ridge, selection["group_best_hgb_candidates"]["all"]]:
    row = by_fold.loc[name]
    ax.plot(row.index, row.values, marker="o", label=name)
ax.set_xticks(by_fold.columns, [f"Fold {fold}" for fold in by_fold.columns])
ax.set_ylabel("Mean absolute error (ms)")
ax.set_title("Validation error by chronological fold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(REPORTS / "validation_regime_change.png", dpi=160)
plt.show()

## 5. Test results

Replay produces a new five-minute forecast at each eligible minute. The fitted
models stay fixed, but earlier test readings become available as lag inputs.

The next cell matches saved predictions to the original target readings and
recomputes the scores. It reports the best test MAE alongside the candidate
already selected by validation; test scores do not change that selection.


In [ ]:
actual_from_source = history.set_index("timestamp")[TARGET].reindex(predictions.target_timestamp).to_numpy()
np.testing.assert_array_equal(actual_from_source, predictions.actual_ms.to_numpy())
metrics = pd.DataFrame([
    {"candidate": name, "selected": name == selected, **score(actual_from_source, predictions[name])}
    for name in selection["test_comparators"]
]).sort_values("mae_ms")
display(metrics[["candidate", "selected", "n", "mae_ms", "rmse_ms", "bias_ms", "p95_absolute_error_ms"]])
print("Best test MAE:", metrics.iloc[0].candidate)
print("Still selected from validation:", selected)
assert predictions.target_timestamp.sub(predictions.forecast_origin).eq(pd.Timedelta(minutes=5)).all()
assert len(predictions) == len(test) - 5
print(f"Scored forecasts: {len(predictions):,}")


## 6. Replay and transition behavior

Plot the selected model, a learned challenger, and the observations. Align each
forecast with its target timestamp rather than its earlier origin. The selected
model may smooth minute-to-minute variation or lag a change in operating level.

The second chart uses 08:30–10:00 on the first complete evaluation day. This fixed
review window lets us inspect transition errors without picking a segment based
on its forecast score.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.8))
ax.plot(predictions.target_timestamp, predictions.actual_ms, linewidth=0.75, alpha=0.55, label="Observed target")
ax.plot(predictions.target_timestamp, predictions[selected], linewidth=1.8, label=f"Selected: {selected}")
ax.plot(predictions.target_timestamp, predictions[best_ridge], linewidth=1.0, alpha=0.8, label="Ridge challenger")
ax.set_ylabel("Aggregate processing time (ms)")
ax.set_xlabel("Target timestamp; source timezone unspecified")
ax.set_title("Fixed-artifact replay: one forecast per minute, five minutes ahead")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%H:%M"))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(REPORTS / "test_forecast_replay.png", dpi=160)
plt.show()

first_complete_day = predictions.target_timestamp.min().normalize() + pd.Timedelta(days=1)
window_start = first_complete_day + pd.Timedelta(hours=8, minutes=30)
window_end = first_complete_day + pd.Timedelta(hours=10)
window = predictions.loc[predictions.target_timestamp.between(window_start, window_end)]
if window.empty:
    raise RuntimeError("The declared transition-review window has no evaluation observations.")
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(window.target_timestamp, window.actual_ms, linewidth=1.0, label="Observed target")
ax.plot(window.target_timestamp, window[selected], linewidth=2, label=f"Selected: {selected}")
ax.plot(window.target_timestamp, window[selection["group_best_hgb_candidates"]["clock"]], linewidth=1.3, label="Clock-only tree challenger")
ax.set_title("Transition review: the first full evaluation day, 08:30–10:00")
ax.set_ylabel("Aggregate processing time (ms)")
ax.set_xlabel("Target timestamp; source timezone unspecified")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(REPORTS / "transition_review.png", dpi=160)
plt.show()

display(pd.read_csv(REPORTS / "forecast_variation.csv"))

## 7. Saved-model inference

`LatencyForecaster` loads the saved model, checks its checksum and dependency
versions, and builds features with the training code. The next cell compares a
prediction from the full history file with one from history cut off at the
forecast origin. They should match.

The shared loader requires 1,441 consecutive readings to support daily lags and
the learned models. A 15-minute trailing mean itself needs only 15 readings, so
this interface asks for more history than that baseline needs.


In [ ]:
forecaster = LatencyForecaster(MODELS)
origin = test.timestamp.iloc[0]
with threadpool_limits(limits=1):
    response = forecaster.predict(history, as_of=origin)
    prefix_only = forecaster.predict(history.loc[history.timestamp.le(origin)], as_of=origin)
assert response == prefix_only
assert np.isclose(response["prediction_ms"], predictions[selected].iloc[0])
display(pd.Series(response, name="saved_artifact_response"))
print("Saved artifacts and roles:")
display(pd.DataFrame(json.loads((MODELS / "registry.json").read_text())))

## 8. Review

Compare the candidate selected by validation with the leader in the test table.
They need not match. Choosing again after viewing test errors would use that
interval for model selection rather than evaluation.

These are point forecasts. They do not establish calibrated prediction intervals,
future-incident detection, or performance on an independent later week.

Run `python tests/test_forecasting.py --data-dir data/raw` from the repository
root to check causal features, fold boundaries, saved artifacts, and reproduced
metrics. The suite also retrains the experiment in a temporary directory.
`docs/forecasting.md` records the modeling decisions.
